# Manual CZ Robustness Optimization

This notebook demonstrates curvature-based robustness optimization for a manually defined CZ gate. The robust cost is

$$
C = 1 - F(\Omega) - \eta \frac{\partial^2 F}{\partial \Omega^2}.
$$

The second derivative is evaluated by a central finite difference,

$$
\frac{\partial^2 F}{\partial \Omega^2} \approx \frac{F(\Omega+\delta)-2F(\Omega)+F(\Omega-\delta)}{\delta^2}.
$$

The same finite-difference combination is applied to the GRAPE gradient, so the robust optimizer can still use gradient-based optimization.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np

project_root = Path.cwd()
if project_root.name == "examples":
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from time_optimal_grape import (
    AveragePhaseGateFidelity,
    ControlLayout,
    ControlValues,
    CurvatureRobustnessSettings,
    GrapeOptimizer,
    GrapeProblem,
    ManualHamiltonianBlock,
    OptimizerSettings,
    RobustGrapeOptimizer,
    assemble_parameters,
    basis_state,
    phase_sample_axis,
    random_control_profile,
    scan_parameter_sensitivity,
    unwrap_phase_profile,
)


## Manual CZ Blocks

The Hamiltonian blocks are the same as in the CZ notebook. The important part for robustness is that `make_problem(omega_value)` can rebuild the full problem at a perturbed Rabi frequency.

In [ ]:
def make_cz_blocks(omega: float, blockade: float) -> tuple[ManualHamiltonianBlock, ...]:
    def h_zero(control_values: ControlValues, sample_index: int) -> np.ndarray:
        return np.zeros((1, 1), dtype=np.complex128)

    def dh_zero(control_values: ControlValues, sample_index: int) -> np.ndarray:
        return np.zeros((1, 1), dtype=np.complex128)

    def h_one(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        drive = omega * np.exp(1j * phi) / 2.0
        matrix = np.zeros((2, 2), dtype=np.complex128)
        matrix[0, 1] = drive
        matrix[1, 0] = np.conj(drive)
        return matrix

    def dh_one(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        drive_derivative = 1j * omega * np.exp(1j * phi) / 2.0
        matrix = np.zeros((2, 2), dtype=np.complex128)
        matrix[0, 1] = drive_derivative
        matrix[1, 0] = np.conj(drive_derivative)
        return matrix

    def h_two(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        drive = np.sqrt(2.0) * omega * np.exp(1j * phi) / 2.0
        matrix = np.zeros((3, 3), dtype=np.complex128)
        matrix[0, 1] = drive
        matrix[1, 0] = np.conj(drive)
        matrix[1, 2] = drive
        matrix[2, 1] = np.conj(drive)
        matrix[2, 2] = blockade
        return matrix

    def dh_two(control_values: ControlValues, sample_index: int) -> np.ndarray:
        phi = control_values.time_controls["phi"][sample_index]
        drive_derivative = 1j * np.sqrt(2.0) * omega * np.exp(1j * phi) / 2.0
        matrix = np.zeros((3, 3), dtype=np.complex128)
        matrix[0, 1] = drive_derivative
        matrix[1, 0] = np.conj(drive_derivative)
        matrix[1, 2] = drive_derivative
        matrix[2, 1] = np.conj(drive_derivative)
        return matrix

    return (
        ManualHamiltonianBlock("00", 1, basis_state(1, 0), basis_state(1, 0), h_zero, {"phi": dh_zero}, 0.0, {"theta": 0.0}),
        ManualHamiltonianBlock("01_or_10", 2, basis_state(2, 0), basis_state(2, 0), h_one, {"phi": dh_one}, 0.0, {"theta": 1.0}),
        ManualHamiltonianBlock("11", 1, basis_state(3, 0), basis_state(3, 0), h_two, {"phi": dh_two}, np.pi, {"theta": 2.0}),
    )


In [ ]:
samples = 40
duration = 8.0
omega = 1.0
blockade = 100.0
omega_delta = 0.01 * omega

layout = ControlLayout(time_control_names=("phi",), local_phase_names=("theta",), samples=samples)
settings = OptimizerSettings(method="BFGS", max_iterations=250, gradient_tolerance=1e-8, display=False)


def make_problem(omega_value: float) -> GrapeProblem:
    return GrapeProblem(
        blocks=make_cz_blocks(omega=omega_value, blockade=blockade),
        control_layout=layout,
        duration=duration,
        objective=AveragePhaseGateFidelity(hilbert_dimension=4),
        optimizer_settings=settings,
    )


rng = np.random.default_rng(21)
initial_parameters = assemble_parameters(
    layout=layout,
    time_controls={"phi": random_control_profile(samples=samples, low=-0.05, high=0.05, rng=rng)},
    local_phases={"theta": 0.0},
)

nominal_problem = make_problem(omega)
plus_problem = make_problem(omega + omega_delta)
minus_problem = make_problem(omega - omega_delta)


## Compare Standard and Robust Optimization

In [ ]:
standard_result = GrapeOptimizer(nominal_problem).optimize(initial_parameters)

robust_settings = CurvatureRobustnessSettings(eta=2e-4, finite_difference_step=omega_delta)
robust_result = RobustGrapeOptimizer(
    nominal_problem=nominal_problem,
    plus_problem=plus_problem,
    minus_problem=minus_problem,
    robustness_settings=robust_settings,
).optimize(initial_parameters)

print(f"standard nominal infidelity: {1.0 - standard_result.fidelity:.6e}")
print(f"robust nominal infidelity: {1.0 - robust_result.fidelity:.6e}")
print(f"robust optimizer cost: {robust_result.infidelity:.6e}")


In [ ]:
time_axis = phase_sample_axis(duration=duration, samples=samples)
standard_controls = layout.unpack(standard_result.parameters)
robust_controls = layout.unpack(robust_result.parameters)

plt.plot(time_axis, unwrap_phase_profile(standard_controls.time_controls["phi"]), label="standard")
plt.plot(time_axis, unwrap_phase_profile(robust_controls.time_controls["phi"]), label="robust")
plt.xlabel(r"dimensionless time $t\Omega$")
plt.ylabel("unwrapped phase")
plt.title("CZ phase profiles")
plt.legend()
plt.grid(True)
plt.show()


## Scan Omega Sensitivity

The profile is fixed during this scan. Only the Hamiltonian amplitude parameter is changed.

In [ ]:
omega_values = omega * np.linspace(0.97, 1.03, 17)
standard_omega_values, standard_infidelities = scan_parameter_sensitivity(
    problem_factory=make_problem,
    parameters=standard_result.parameters,
    parameter_values=omega_values,
)
robust_omega_values, robust_infidelities = scan_parameter_sensitivity(
    problem_factory=make_problem,
    parameters=robust_result.parameters,
    parameter_values=omega_values,
)

plt.semilogy(standard_omega_values / omega, standard_infidelities, marker="o", label="standard")
plt.semilogy(robust_omega_values / omega, robust_infidelities, marker="o", label="robust")
plt.xlabel(r"relative Rabi frequency $\Omega'/\Omega$")
plt.ylabel("infidelity")
plt.title("CZ Omega robustness scan")
plt.legend()
plt.grid(True)
plt.show()
